#### RESUMEN DE VARIABLES CATEGÓRICAS IDENTIFICADAS
Variables tipo objeto (string)
|Variable|Valores Únicos|Observaciones|
|--------|--------------|-------------|
|description|11,804|Texto libre, muy alta cardinalidad|
|homeType|8|Tipos de propiedad: SINGLE_FAMILY, CONDO, TOWNHOUSE, etc.|

Variables numéricas con potencial categórico
|Variable|Cardinalidad|Tipo sugerido|
|--------|------------|-------------|
|bedrooms|11|Ordinal|
|bathrooms|20|Ordinal|
|zip_3digit|16|Nominal geográfico|
|zipcode|64|Nominal geográfico (alta cardinalidad)|
|max_school_rating|8|Ordinal (3-10)|
|propertyTaxRate|16|Continua o binning|
|Variables binarias (has_pool, has_garage, etc.)|2|✅ Ya están listas (0/1)|

In [1]:
import pandas as pd
import numpy as np

# ============================================================
# 1. CARGAR EL DATASET
# ============================================================
df = pd.read_csv(r"C:\Users\Ing. Antonio Rial\OneDrive - Universidad Austral\MCD_Laboratorio.de.Implementación.2\Competencia.House.Pricing\data\tabular\train_processed.csv", low_memory=False)


🎯 RECOMENDACIONES DE ENCODING  
✅ ONE-HOT ENCODING  
Para variables nominales de baja cardinalidad (<10 valores)  

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 📥 1. Cargar dataset
#df = pd.read_csv('train_processed.csv')
print(f"📊 Dimensiones originales: {df.shape}")

# 🔧 2. Preprocesamiento previo: agrupar categorías raras en homeType
# (Evita que el modelo aprenda ruido de tipos con muy pocas muestras)
UMBRAL_RAREZAS = 50
conteos = df['homeType'].value_counts()
categorias_raras = conteos[conteos < UMBRAL_RAREZAS].index
df['homeType'] = df['homeType'].replace(categorias_raras, 'OTHER')

# 🎯 3. Definir variables para One-Hot Encoding
cols_ohe = ['homeType', 'zip_3digit']

# ==========================================================
# 🔹 OPCIÓN A: Rápida con pandas (ideal para EDA / notebooks)
# ==========================================================
df_ohe_pandas = pd.get_dummies(
    df, 
    columns=cols_ohe, 
    drop_first=True,  # Evita multicolinealidad
    dtype=int         # Genera 0/1 en lugar de True/False
)

print("\n📦 OPCIÓN A (pandas.get_dummies)")
print(f"   Dimensiones tras OHE: {df_ohe_pandas.shape}")
print("   Nuevas columnas:", 
      [c for c in df_ohe_pandas.columns if c.startswith(('homeType_', 'zip_3digit_'))][:10], "...")

# ==========================================================
# 🔹 OPCIÓN B: Robusta con scikit-learn (recomendada para ML)
# ==========================================================
# Maneja automáticamente valores faltantes, categorías desconocidas y se integra en pipelines
encoder = OneHotEncoder(
    sparse_output=False,  # Devuelve array numpy en lugar de matriz dispersa
    drop='first',         # Evita multicolinealidad
    handle_unknown='ignore', # Ignora categorías no vistas en train
    dtype=int
)

# Crear transformador de columnas
preprocessor = ColumnTransformer(
    transformers=[('ohe', encoder, cols_ohe)],
    remainder='passthrough',  # Mantiene el resto de columnas intactas
    n_jobs=-1
)

# Ajustar y transformar
array_encoded = preprocessor.fit_transform(df)

# Reconstruir DataFrame con nombres de columnas correctos
ohe_feature_names = preprocessor.named_transformers_['ohe'].get_feature_names_out(cols_ohe)
otros_cols = [c for c in df.columns if c not in cols_ohe]
nuevos_cols = list(ohe_feature_names) + otros_cols

df_ohe_sklearn = pd.DataFrame(array_encoded, columns=nuevos_cols, index=df.index)

print("\n📦 OPCIÓN B (sklearn.OneHotEncoder)")
print(f"   Dimensiones tras OHE: {df_ohe_sklearn.shape}")
print("   Primeras 5 columnas generadas:", nuevos_cols[:5])

# 💡 Guardar resultado (descomenta cuando lo necesites)
# df_ohe_sklearn.to_csv('train_processed_ohe.csv', index=False)

📊 Dimensiones originales: (11840, 46)

📦 OPCIÓN A (pandas.get_dummies)
   Dimensiones tras OHE: (11840, 64)
   Nuevas columnas: ['homeType_CONDO', 'homeType_MULTI_FAMILY', 'homeType_OTHER', 'homeType_SINGLE_FAMILY', 'homeType_TOWNHOUSE', 'zip_3digit_323', 'zip_3digit_324', 'zip_3digit_328', 'zip_3digit_330', 'zip_3digit_331'] ...

📦 OPCIÓN B (sklearn.OneHotEncoder)
   Dimensiones tras OHE: (11840, 64)
   Primeras 5 columnas generadas: ['homeType_CONDO', 'homeType_MULTI_FAMILY', 'homeType_OTHER', 'homeType_SINGLE_FAMILY', 'homeType_TOWNHOUSE']


#### Notas clave para one-hot encoding:  
|Aspecto|Recomendación|
|-------|-------------|
|drop_first=True|Elimina la primera categoría dummy para evitar multicolinealidad en modelos lineales (Regresión, Logistic, SVM, etc.).|
|handle_unknown='ignore'|Crucial en producción: si aparece un zip_3digit o homeType nuevo en test/producción, no romperá el modelo.|
|Agrupación de homeType|Categorías como LOT, MANUFACTURED o HOME_TYPE_UNKNOWN suelen tener <50 registros. Agruparlas en OTHER mejora la estabilidad del encoding.|
|zip_3digit vs zipcode|zip_3digit tiene 16 valores → ideal para OHE. zipcode (64 valores) te recomiendo Target Encoding para no inflar la dimensionalidad.|